# D3 · Reading a 10x10 confusion matrix with two rare classes

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 45 minutes · **core path ★**

## Objective

Interpret the metrics of a multiclass classifier when two classes are rare in the training set: precision, recall and F1 **per class**, micro, macro and weighted average, and understand where
the errors of the rare classes end up. Then fix with `class_weight="balanced"`.

## Dataset

`load_digits` (60/40 split) where in the training set the digits **3** and **8** were reduced by 85% (about 16 examples each remain against 107 for the others). The test set is balanced like the original:
`X_s, y_s` (imbalanced training set), `X_te, y_te`.

## Educational purpose

Learn to read a multiclass confusion matrix and to recognize the classic sign of rare classes: **high precision and low recall** (the model predicts them rarely, but when it does it is right). See what
micro and macro hide.

## Task

1. Train `model` = `StandardScaler` + `LogisticRegression(C=0.1, max_iter=3000)` on `X_s, y_s` and save `pred`, `cm` (confusion matrix of the test set) and `class_table` (DataFrame index 0..9, columns `precision`, `recall`, `f1`, `support` with `precision_recall_fscore_support`).
2. Save `accuracy`, `micro_f1`, `macro_f1`, `weighted_f1` and `micro_equals_accuracy` (True if micro F1 coincides with the accuracy: it is a general fact for single-label multiclass).
3. Find `silent_classes`: the (sorted) list of the classes with `precision − recall > 0.15`. For each one find in `where_they_go` (dictionary) the **wrong** class into which its digits are most often classified (the column with the highest count, excluding the diagonal).
4. Redo with `class_weight="balanced"`: save `rare_recall_before` and `rare_recall_after` (mean recall of the two rare classes), `balanced_macro_f1` and `balanced_accuracy`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.4, stratify=digits.target, random_state=0)
rng = np.random.default_rng(0)
keep = np.ones(len(y_tr), dtype=bool)
for c in (3, 8):
    idx = np.where(y_tr == c)[0]
    keep[rng.choice(idx, int(0.85 * len(idx)), replace=False)] = False
X_s, y_s = X_tr[keep], y_tr[keep]
print("examples per class in the training set:", np.bincount(y_s))

## Your work

In [ ]:
model = None
pred = None
cm = None
class_table = None

In [ ]:
accuracy = None
micro_f1 = None
macro_f1 = None
weighted_f1 = None
micro_equals_accuracy = None

In [ ]:
silent_classes = None
where_they_go = None

In [ ]:
rare_recall_before = None
rare_recall_after = None
balanced_macro_f1 = None
balanced_accuracy = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`precision, recall, f1, support = precision_recall_fscore_support(y_te, pred)`. For class c: `errors = cm[c].copy(); errors[c] = 0; np.argmax(errors)`.'):
    check(cm.shape == (10, 10) and cm.sum() == len(y_te) and list(class_table.columns) == ["precision", "recall", "f1", "support"], "10x10 matrix and per-class table", "confusion_matrix(y_te, pred), precision_recall_fscore_support")
    check(close(accuracy, (pred == y_te).mean()) and close(macro_f1, f1_score(y_te, pred, average="macro")) and close(weighted_f1, f1_score(y_te, pred, average="weighted")), "correct accuracy, macro and weighted F1", "average='macro' and 'weighted'")
    check(micro_equals_accuracy is True and close(micro_f1, accuracy), "micro F1 = accuracy", "in multiclass with a single label per example, micro-F1 = accuracy")
    check(silent_classes == [3, 8], "the silent classes are 3 and 8: high precision, low recall", "precision - recall > 0.15")
    check(all(where_they_go[c] == int(np.argmax(np.where(np.eye(10, dtype=bool)[c], -1, cm[c]))) and where_they_go[c] != c for c in silent_classes), "for each silent class, the most frequent wrong class is given", "row of the class in the confusion matrix, without the diagonal")
    check(rare_recall_after > rare_recall_before + 0.05, "with class_weight='balanced' the recall of the rare classes goes up by over 0.05", "mean recall of classes 3 and 8")
    check(balanced_macro_f1 > macro_f1 and balanced_accuracy > accuracy, "and macro F1 and accuracy improve too", "compare the values before and after")
    check(macro_f1 < accuracy, "macro F1 is lower than the accuracy: the rare classes weigh more in the macro", "compare macro_f1 and accuracy")
done()

---
The commented solution is in `solutions/D_interpret/D3_multiclass_metrics_with_rare_classes.ipynb`. Open it after you pass the check, or after you have really tried.